# FeatureBuilder3.0 REPO FIX

**Purpose:** Build the final quarterly research dataset for the MBA Finance study on Indian monsoon outcomes, macroeconomic indicators, and short-term allocation between mid-cap and large-cap equities.

The notebook prepares one clean model input file:

`processed/quarterly_features_clean.csv`

It also writes standardized intermediate CSVs, a data dictionary, and a manifest file for auditability.


## Expected raw input files

Place the raw files in the `raw/` folder before running the notebook.

The notebook is written to tolerate minor column-name differences, but the following file names are expected by default:

| Dataset | Expected file name |
|---|---|
| CPI monthly index | `CPI_Monthly_Jan_2013_to_Jun_2025.csv` |
| GDP quarterly level | `GDP_Quarterly_2010_2025.csv` |
| RBI repo rate | `Repo_Rate_Monthly_2010_2025.csv` |
| Mid-cap index price/NAV | `NIFTYMidcap100.csv` |
| Large-cap index price/NAV | `Nifty50.csv` |
| Monthly rainfall | `AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv` |

If your file names differ, edit the `RAW_FILES` dictionary in the next cell.


In [1]:
# =========================
# 0) Imports, folders, configuration
# =========================

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RAW = Path("./raw")
PROC = Path("./processed")
PROC.mkdir(parents=True, exist_ok=True)

# Edit here if your raw file names are different.
RAW_FILES = {
    "cpi": "CPI_Monthly_Jan_2013_to_Jun_2025.csv",
    "gdp": "GDP_Quarterly_2010_2025.csv",
    "repo": "Repo_Rate_Monthly_2010_2025.csv",
    "midcap": "NIFTYMidcap100.csv",
    "largecap": "Nifty50.csv",
    "rainfall": "AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv",
}


# Fallback aliases: the notebook will use the first file it finds in raw/.
# This prevents failures when a prior notebook expected older names such as Repo_Rate_Monthly_2010_2025.csv.
RAW_FILE_ALIASES = {
    "cpi": ["CPI_Monthly_Jan_2013_to_Jun_2025.csv", "cpi.csv", "CPI.csv"],
    "gdp": ["GDP_Quarterly_2010_2025.csv", "gdp.csv", "GDP.csv"],
    "repo": ["Repo_Rate_Monthly_2010_2025.csv", "Repo_Rate.csv", "repo.csv"],
    "midcap": ["NIFTYMidcap100.csv", "midcap_index.csv", "quarterly_midcap_qret.csv"],
    "largecap": ["Nifty50.csv", "largecap_index.csv", "quarterly_nifty_qret.csv"],
    "rainfall": ["AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv", "rainfall.csv"],
}

# Monsoon classification threshold used for RQ2.
# +5% or above = favorable/good monsoon; -5% or below = poor monsoon.
MONSOON_THRESHOLD_PCT = 5.0

print("Raw folder:", RAW.resolve())
print("Processed folder:", PROC.resolve())


Raw folder: C:\Users\Local User\Documents\GitHub\marketpredict\raw
Processed folder: C:\Users\Local User\Documents\GitHub\marketpredict\processed


In [2]:
# =========================
# 1) Helper functions
# =========================

def pick_col(df: pd.DataFrame, candidates, required=True, label="column"):
    """Return the first matching column using exact then loose case-insensitive matching."""
    cols = list(df.columns)
    lower_map = {str(c).lower().strip(): c for c in cols}

    for cand in candidates:
        key = str(cand).lower().strip()
        if key in lower_map:
            return lower_map[key]

    for col in cols:
        col_l = str(col).lower().strip()
        for cand in candidates:
            cand_l = str(cand).lower().strip()
            if cand_l in col_l or col_l in cand_l:
                return col

    if required:
        raise KeyError(f"Could not identify {label}. Candidates={candidates}. Available columns={cols}")
    return None


def smart_parse_dates(s: pd.Series) -> pd.Series:
    """Parse mixed date strings robustly."""
    s = s.astype(str).str.strip()
    attempts = [
        pd.to_datetime(s, errors="coerce"),
        pd.to_datetime(s, errors="coerce", dayfirst=True),
    ]
    fmts = [
        "%m/%d/%Y", "%d/%m/%Y", "%d-%m-%Y", "%d-%b-%Y", "%d-%b-%y", "%Y-%m-%d",
        "%b %d, %Y", "%d %b %Y", "%b %Y", "%B %Y", "%m-%Y", "%Y/%m/%d",
    ]
    attempts.extend(pd.to_datetime(s, format=fmt, errors="coerce") for fmt in fmts)

    def score(dt):
        ok = dt.dropna()
        if ok.empty:
            return (0, 0)
        return (len(ok), ok.dt.year.nunique())

    return max(attempts, key=score)


def to_numeric_clean(x: pd.Series) -> pd.Series:
    """Convert strings with commas/percent signs to numeric values."""
    return pd.to_numeric(
        x.astype(str)
         .str.replace(",", "", regex=False)
         .str.replace("%", "", regex=False)
         .str.strip(),
        errors="coerce",
    )


def standardize_q_series(s: pd.Series, name: str) -> pd.DataFrame:
    """Convert a quarterly series indexed by dates into a standard quarter-labeled DataFrame."""
    s = s.dropna().copy()
    s.index = pd.to_datetime(s.index).to_period("Q").to_timestamp("Q")
    s = s[~s.index.duplicated(keep="last")].sort_index()
    per = s.index.to_period("Q")
    return pd.DataFrame({
        "quarter_end": s.index,
        "year": per.year,
        "quarter": per.quarter,
        "quarter_label": [f"{y}-Q{q}" for y, q in zip(per.year, per.quarter)],
        name: s.values,
    })



def raw_path(key: str) -> Path:
    """Return the raw file path for a dataset key, using aliases if needed."""
    candidates = []
    if key in RAW_FILES:
        candidates.append(RAW_FILES[key])
    candidates.extend(RAW_FILE_ALIASES.get(key, []))

    # De-duplicate while preserving order
    seen = set()
    candidates = [c for c in candidates if not (c in seen or seen.add(c))]

    for name in candidates:
        path = RAW / name
        if path.exists():
            return path

    available = sorted([p.name for p in RAW.glob("*")]) if RAW.exists() else []
    raise FileNotFoundError(
        f"Missing raw file for '{key}'. Tried: {candidates}. "
        f"Available files in {RAW}: {available}"
    )


def read_csv_required(path: Path) -> pd.DataFrame:
    """Read CSV and show a useful error with available raw files."""
    path = Path(path)
    if not path.exists():
        available = sorted([p.name for p in RAW.glob("*.csv")]) if RAW.exists() else []
        raise FileNotFoundError(
            f"Missing file: {path}. Available CSV files in {RAW}: {available}. "
            "Check RAW_FILES in the configuration cell or use raw_path('<key>')."
        )
    return pd.read_csv(path)


def save_standard(df: pd.DataFrame, filename: str):
    path = PROC / filename
    df.to_csv(path, index=False, float_format="%.6f")
    print("Saved", path.resolve())
    return path


def load_standard_series(filename: str) -> pd.Series:
    path = PROC / filename
    df = pd.read_csv(path, parse_dates=["quarter_end"])
    key_cols = {"quarter_end", "year", "quarter", "quarter_label"}
    val_cols = [c for c in df.columns if c not in key_cols]
    if len(val_cols) != 1:
        raise ValueError(f"{filename}: expected exactly one value column, found {val_cols}")
    name = val_cols[0]
    s = pd.Series(df[name].values, index=df["quarter_end"], name=name)
    s.index = s.index.to_period("Q").to_timestamp("Q")
    return s.sort_index()


In [3]:
# =========================
# 1B) Raw file resolution check
# =========================

print("Available files in raw folder:")
for p in sorted(RAW.glob("*.csv")):
    print(" -", p.name)

print("\nResolved raw input files:")
for key in ["cpi", "gdp", "repo", "midcap", "largecap", "rainfall"]:
    print(f"{key:10s} -> {raw_path(key)}")


Available files in raw folder:
 - AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv
 - CPI_Monthly_Jan_2013_to_Jun_2025.csv
 - GDP_Quarterly_2010_2025.csv
 - Nifty50.csv
 - NIFTYMidcap100.csv
 - quarterly_cpi_yoy.csv
 - quarterly_excess_ret.csv
 - quarterly_gdp_yoy.csv
 - quarterly_merged_intersection.csv
 - quarterly_merged_outer.csv
 - quarterly_midcap_qret.csv
 - quarterly_nifty_qret.csv
 - quarterly_repo_chg_bps.csv
 - quarterly_repo_level.csv
 - Repo_Rate_Monthly_2010_2025.csv

Resolved raw input files:
cpi        -> raw\CPI_Monthly_Jan_2013_to_Jun_2025.csv
gdp        -> raw\GDP_Quarterly_2010_2025.csv
repo       -> raw\Repo_Rate_Monthly_2010_2025.csv
midcap     -> raw\NIFTYMidcap100.csv
largecap   -> raw\Nifty50.csv
rainfall   -> raw\AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv


## Build standardized quarterly input series

Each section writes one standardized file into `processed/` with these keys:

`quarter_end`, `year`, `quarter`, `quarter_label`, and one value column.


In [4]:
# =========================
# 2) CPI: monthly index -> YoY percentage -> quarterly mean
# =========================

cpi_raw = read_csv_required(raw_path("cpi"))

c_date = pick_col(cpi_raw, ["DATE", "Date", "Month", "Period"], label="CPI date column")
c_val = pick_col(cpi_raw, ["CPI", "Index", "Value", "CPI_COMBINED_RAW2012_100"], label="CPI value column")

cpi = cpi_raw.copy()
cpi[c_date] = smart_parse_dates(cpi[c_date])
cpi[c_val] = to_numeric_clean(cpi[c_val])
cpi = cpi.dropna(subset=[c_date, c_val]).sort_values(c_date)

cpi_m = cpi.set_index(c_date)[c_val].sort_index()
cpi_m.index = pd.to_datetime(cpi_m.index)
cpi_m = cpi_m.resample("M").last().ffill()

cpi_yoy_m = cpi_m.pct_change(12) * 100.0
cpi_yoy_q = cpi_yoy_m.resample("Q").mean().rename("cpi_yoy")

cpi_df = standardize_q_series(cpi_yoy_q, "cpi_yoy")
save_standard(cpi_df, "quarterly_cpi_yoy.csv")

print(cpi_df.head())
print(cpi_df.tail())


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_cpi_yoy.csv
  quarter_end  year  quarter quarter_label   cpi_yoy
0  2014-03-31  2014        1       2014-Q1  8.244298
1  2014-06-30  2014        2       2014-Q2  7.859486
2  2014-09-30  2014        3       2014-Q3  6.681568
3  2014-12-31  2014        4       2014-Q4  4.054538
4  2015-03-31  2015        1       2015-Q1  5.272440
   quarter_end  year  quarter quarter_label   cpi_yoy
41  2024-06-30  2024        2       2024-Q2  4.904469
42  2024-09-30  2024        3       2024-Q3  4.244829
43  2024-12-31  2024        4       2024-Q4  5.634890
44  2025-03-31  2025        1       2025-Q1  3.733903
45  2025-06-30  2025        2       2025-Q2  2.695618


In [5]:
# =========================
# 3) GDP: quarterly level -> YoY percentage
# =========================

gdp_raw = read_csv_required(raw_path("gdp"))

gdp_q_col = pick_col(gdp_raw, ["quarter", "Quarter", "period", "Period", "date", "Date"], label="GDP quarter/date column")
gdp_val_col = pick_col(gdp_raw, ["gdp", "GDP", "value", "Value", "level", "Level"], label="GDP value column")

gdp = gdp_raw.copy()
gdp[gdp_val_col] = to_numeric_clean(gdp[gdp_val_col])

qtext = gdp[gdp_q_col].astype(str).str.strip()

# Supports strings like 2014Q1, 2014-Q1, FY2014 Q1, etc.
extract = qtext.str.extract(r"(?P<year>\d{4}).*?[Qq](?P<quarter>[1-4])")
valid_q = extract["year"].notna() & extract["quarter"].notna()

if valid_q.any():
    qstr = extract.loc[valid_q, "year"] + "Q" + extract.loc[valid_q, "quarter"]
    # Indian GDP is commonly reported using fiscal quarters. Q-MAR maps FY-style quarters to dates ending Mar/Jun/Sep/Dec.
    q_end = pd.PeriodIndex(qstr, freq="Q-MAR").to_timestamp(how="end").to_period("Q").to_timestamp("Q")
    gdp_level_q = pd.Series(gdp.loc[valid_q, gdp_val_col].values, index=q_end).sort_index()
else:
    # Fallback: parse as date and resample to quarter-end.
    gdp[gdp_q_col] = smart_parse_dates(gdp[gdp_q_col])
    gdp = gdp.dropna(subset=[gdp_q_col, gdp_val_col]).sort_values(gdp_q_col)
    gdp_level_q = gdp.set_index(gdp_q_col)[gdp_val_col].resample("Q").last()

gdp_level_q = gdp_level_q[~gdp_level_q.index.duplicated(keep="last")].sort_index()
gdp_yoy_q = gdp_level_q.pct_change(4).mul(100.0).rename("gdp_yoy")

gdp_df = standardize_q_series(gdp_yoy_q, "gdp_yoy")
save_standard(gdp_df, "quarterly_gdp_yoy.csv")

print(gdp_df.head())
print(gdp_df.tail())


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_gdp_yoy.csv
  quarter_end  year  quarter quarter_label   gdp_yoy
0  2012-06-30  2012        2       2012-Q2  4.296010
1  2012-09-30  2012        3       2012-Q3  6.447099
2  2012-12-31  2012        4       2012-Q4  7.337749
3  2013-03-31  2013        1       2013-Q1  6.534983
4  2013-06-30  2013        2       2013-Q2  5.342980
   quarter_end  year  quarter quarter_label   gdp_yoy
44  2023-06-30  2023        2       2023-Q2  8.354641
45  2023-09-30  2023        3       2023-Q3  6.511800
46  2023-12-31  2023        4       2023-Q4  5.612025
47  2024-03-31  2024        1       2024-Q1  6.366486
48  2024-06-30  2024        2       2024-Q2  7.384530


In [6]:
# =========================
# 4) RBI repo rate: level and quarterly change in basis points
# =========================

repo_raw = read_csv_required(raw_path("repo"))

r_date = pick_col(repo_raw, ["DATE", "Date", "date", "Month", "Period"], label="repo date column")
r_val = pick_col(repo_raw, ["repo", "Repo", "repo_rate", "Repo Rate", "rate", "Rate", "Value"], label="repo value column")

repo = repo_raw.copy()
repo[r_date] = smart_parse_dates(repo[r_date])
repo[r_val] = to_numeric_clean(repo[r_val])
repo = repo.dropna(subset=[r_date, r_val]).sort_values(r_date)

repo_m = repo.set_index(r_date)[r_val].sort_index()
repo_m.index = pd.to_datetime(repo_m.index)
repo_m = repo_m.resample("M").last().ffill()

repo_q = repo_m.resample("Q").last().rename("repo")
repo_chg_bps_q = repo_q.diff().mul(100.0).rename("repo_chg_bps")

repo_df = standardize_q_series(repo_q, "repo")
repo_chg_df = standardize_q_series(repo_chg_bps_q, "repo_chg_bps")

save_standard(repo_df, "quarterly_repo_level.csv")
save_standard(repo_chg_df, "quarterly_repo_chg_bps.csv")

print(repo_df.head())
print(repo_chg_df.head())


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_repo_level.csv
Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_repo_chg_bps.csv
  quarter_end  year  quarter quarter_label  repo
0  2010-03-31  2010        1       2010-Q1  5.00
1  2010-06-30  2010        2       2010-Q2  5.25
2  2010-09-30  2010        3       2010-Q3  6.00
3  2010-12-31  2010        4       2010-Q4  6.25
4  2011-03-31  2011        1       2011-Q1  6.75
  quarter_end  year  quarter quarter_label  repo_chg_bps
0  2010-06-30  2010        2       2010-Q2          25.0
1  2010-09-30  2010        3       2010-Q3          75.0
2  2010-12-31  2010        4       2010-Q4          25.0
3  2011-03-31  2011        1       2011-Q1          50.0
4  2011-06-30  2011        2       2011-Q2          75.0


In [7]:
# =========================
# 5) Equity returns: mid-cap versus large-cap relative quarterly performance
# =========================

mid_raw = read_csv_required(raw_path("midcap"))
large_raw = read_csv_required(raw_path("largecap"))

def build_price_series(raw: pd.DataFrame, label: str) -> pd.Series:
    date_col = pick_col(raw, ["Date", "DATE", "date", "timestamp", "Timestamp"], label=f"{label} date column")
    price_col = pick_col(raw, ["Close", "close", "Adj Close", "adj_close", "Price", "price", "Value", "value", "NAV", "nav"], label=f"{label} price/value column")
    df = raw.copy()
    # The uploaded Nifty files use MM/DD/YYYY monthly dates such as 08/01/2025 = Aug 1, 2025.
    parsed_mdy = pd.to_datetime(df[date_col].astype(str).str.strip(), format="%m/%d/%Y", errors="coerce")
    if parsed_mdy.notna().mean() >= 0.80:
        df[date_col] = parsed_mdy
    else:
        df[date_col] = smart_parse_dates(df[date_col])
    df[price_col] = to_numeric_clean(df[price_col])
    df = df.dropna(subset=[date_col, price_col]).sort_values(date_col)
    s = df.set_index(date_col)[price_col]
    s.index = pd.to_datetime(s.index)
    return s[~s.index.duplicated(keep="last")].sort_index().rename(label)

mid_price = build_price_series(mid_raw, "midcap_price")
large_price = build_price_series(large_raw, "largecap_price")

# Quarter-end prices and quarterly simple returns.
mid_q_price = mid_price.resample("Q").last()
large_q_price = large_price.resample("Q").last()

midcap_ret = mid_q_price.pct_change().mul(100.0).rename("midcap_ret")
largecap_ret = large_q_price.pct_change().mul(100.0).rename("largecap_ret")
excess_ret = (midcap_ret - largecap_ret).rename("excess_ret")

midcap_ret_df = standardize_q_series(midcap_ret, "midcap_ret")
largecap_ret_df = standardize_q_series(largecap_ret, "largecap_ret")
excess_ret_df = standardize_q_series(excess_ret, "excess_ret")

save_standard(midcap_ret_df, "quarterly_midcap_ret.csv")
save_standard(largecap_ret_df, "quarterly_largecap_ret.csv")
save_standard(excess_ret_df, "quarterly_excess_ret.csv")

print(excess_ret_df.head())
print(excess_ret_df.tail())


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_midcap_ret.csv
Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_largecap_ret.csv
Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_excess_ret.csv
  quarter_end  year  quarter quarter_label  excess_ret
0  2010-06-30  2010        2       2010-Q2    4.320474
1  2010-09-30  2010        3       2010-Q3   -0.795323
2  2010-12-31  2010        4       2010-Q4   -5.084365
3  2011-03-31  2011        1       2011-Q1   -4.322099
4  2011-06-30  2011        2       2011-Q2    2.340503
   quarter_end  year  quarter quarter_label  excess_ret
57  2024-09-30  2024        3       2024-Q3    0.426823
58  2024-12-31  2024        4       2024-Q4    3.480686
59  2025-03-31  2025        1       2025-Q1   -9.132469
60  2025-06-30  2025        2       2025-Q2    7.121778
61  2025-09-30  2025        3       2025-Q3   -1.466165


In [8]:
# =========================
# 6) Rainfall: southwest monsoon anomaly percentage
# =========================

rain_raw = read_csv_required(raw_path("rainfall"))

rain = rain_raw.copy()
y_col = pick_col(rain, ["year", "Year", "YEAR"], label="rainfall year column")
m_col = pick_col(rain, ["month", "Month", "MONTH"], label="rainfall month column")
obs_col = pick_col(rain, ["rainfall_mm", "actual", "observed", "rainfall", "Rainfall"], label="observed rainfall column")
norm_col = pick_col(rain, ["good_rainfall_mm", "normal", "normal_rainfall", "long_period_average", "lpa"], label="normal/good rainfall column")
anom_col = pick_col(rain, ["anomaly_mm", "anom_mm", "anomaly"], required=False, label="rainfall anomaly column")

month_map = {m.lower()[:3]: i for i, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

rain["year_num"] = to_numeric_clean(rain[y_col]).astype("Int64")

if pd.api.types.is_numeric_dtype(rain[m_col]):
    rain["month_num"] = to_numeric_clean(rain[m_col]).astype("Int64")
else:
    rain["month_num"] = rain[m_col].astype(str).str.strip().str[:3].str.lower().map(month_map).astype("Int64")

rain["obs_mm"] = to_numeric_clean(rain[obs_col])
rain["norm_mm"] = to_numeric_clean(rain[norm_col])
if anom_col is not None:
    rain["anom_mm"] = to_numeric_clean(rain[anom_col])
else:
    rain["anom_mm"] = np.nan

missing_months = rain.loc[rain["month_num"].isna(), m_col].dropna().unique().tolist()
if missing_months:
    raise ValueError(f"Unrecognized rainfall month values: {missing_months}")

# Southwest monsoon months: June to September.
mons = rain[rain["month_num"].between(6, 9)].copy()

coverage = mons.groupby("year_num")["month_num"].nunique().rename("monsoon_months").reset_index()
incomplete = coverage[coverage["monsoon_months"] < 4]
if not incomplete.empty:
    print("WARNING: Some years have fewer than four monsoon months:")
    display(incomplete)

season = mons.groupby("year_num", as_index=False).agg(
    obs_mm=("obs_mm", "sum"),
    norm_mm=("norm_mm", "sum"),
    anom_sum_mm=("anom_mm", "sum"),
    monsoon_months=("month_num", "nunique"),
)
season.loc[season["norm_mm"] == 0, "norm_mm"] = np.nan
season["rain_anom"] = ((season["obs_mm"] - season["norm_mm"]) / season["norm_mm"]) * 100.0

# Stamp annual monsoon anomaly at September quarter end, then make it available over following quarters via forward-fill.
rain_idx = pd.to_datetime(season["year_num"].astype(int).astype(str) + "-09-30")
rain_q = pd.Series(season["rain_anom"].values, index=rain_idx, name="rain_anom").sort_index().resample("Q").ffill()

rain_df = standardize_q_series(rain_q, "rain_anom")

save_standard(season.rename(columns={"year_num": "year"}), "rainfall_seasonal_totals.csv")
save_standard(rain_df, "quarterly_rain_anom.csv")

print(season.head())
print(rain_df.head(12))
print(rain_df.tail(12))


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\rainfall_seasonal_totals.csv
Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_rain_anom.csv
   year_num  obs_mm  norm_mm  anom_sum_mm  monsoon_months  rain_anom
0      2012  1061.5    868.5       -193.0               4  22.222222
1      2013  1475.4    868.5       -606.9               4  69.879102
2      2014   803.8    868.5         64.7               4  -7.449626
3      2015   814.9    868.5         53.6               4  -6.171560
4      2016  1347.9    868.5       -479.4               4  55.198618
   quarter_end  year  quarter quarter_label  rain_anom
0   2012-09-30  2012        3       2012-Q3  22.222222
1   2012-12-31  2012        4       2012-Q4  22.222222
2   2013-03-31  2013        1       2013-Q1  22.222222
3   2013-06-30  2013        2       2013-Q2  22.222222
4   2013-09-30  2013        3       2013-Q3  69.879102
5   2013-12-31  2013        4       2013-Q4  69.879102
6   2014-03-31

## Merge into the final study dataset

The final dataset separates:

- **Current-quarter observed variables** such as CPI, GDP, repo-rate change, and rainfall anomaly.
- **Lagged predictors** such as prior-quarter excess return and prior-quarter macro/rainfall values.
- **Target variable**: `next_q_excess_ret`, which is next-quarter mid-cap return minus next-quarter large-cap return.


In [9]:
# =========================
# 7) Merge standardized quarterly files
# =========================

series_files = {
    "midcap_ret": "quarterly_midcap_ret.csv",
    "largecap_ret": "quarterly_largecap_ret.csv",
    "excess_ret": "quarterly_excess_ret.csv",
    "cpi_yoy": "quarterly_cpi_yoy.csv",
    "gdp_yoy": "quarterly_gdp_yoy.csv",
    "repo": "quarterly_repo_level.csv",
    "repo_chg_bps": "quarterly_repo_chg_bps.csv",
    "rain_anom": "quarterly_rain_anom.csv",
}

series = {}
for name, filename in series_files.items():
    try:
        series[name] = load_standard_series(filename)
    except Exception as exc:
        print(f"WARNING: Could not load {filename}: {exc}")

if "excess_ret" not in series:
    raise RuntimeError("quarterly_excess_ret.csv is required because excess_ret is the core dependent series.")

# Inner join keeps only quarters where all available core inputs are aligned.
df_all = pd.concat(series.values(), axis=1, join="inner").sort_index()
df_all.index.name = "quarter_end"

print("Merged quarterly rows:", len(df_all))
print("Date range:", df_all.index.min(), "to", df_all.index.max())
print("Columns:", list(df_all.columns))

df_all_preview = df_all.copy()
df_all_preview.head()


Merged quarterly rows: 42
Date range: 2014-03-31 00:00:00 to 2024-06-30 00:00:00
Columns: ['midcap_ret', 'largecap_ret', 'excess_ret', 'cpi_yoy', 'gdp_yoy', 'repo', 'repo_chg_bps', 'rain_anom']


,midcap_ret,largecap_ret,excess_ret,cpi_yoy,gdp_yoy,repo,repo_chg_bps,rain_anom
quarter_end,,,,,,,,
2014-03-31,6.704620,6.348350,0.356270,8.244298,5.922736,8.0,25.0,69.879102
2014-06-30,28.847192,13.531070,15.316122,7.859486,7.112080,8.0,0.0,69.879102
2014-09-30,2.896304,4.643723,-1.747418,6.681568,7.592544,8.0,0.0,-7.449626
2014-12-31,10.207737,3.991312,6.216425,4.054538,8.033806,8.0,0.0,-7.449626
2015-03-31,3.316950,2.514880,0.802069,5.272440,7.197433,7.5,-50.0,-7.449626


In [10]:
# =========================
# 8) Create study features, target variable, flags, and labels
# =========================

features = df_all.copy().sort_index()

# Primary study target: next-quarter mid-cap minus large-cap relative return.
features["next_q_excess_ret"] = features["excess_ret"].shift(-1)

# Momentum benchmark predictors.
features["lag_excess_ret_1q"] = features["excess_ret"].shift(1)
features["lag_excess_ret_2q"] = features["excess_ret"].shift(2)

# Lagged macroeconomic and rainfall predictors.
for col in ["cpi_yoy", "gdp_yoy", "repo", "repo_chg_bps", "rain_anom"]:
    if col in features.columns:
        features[f"lag_{col}_1q"] = features[col].shift(1)

# Monsoon classification flags for RQ2.
if "rain_anom" in features.columns:
    features["good_monsoon_flag"] = (features["rain_anom"] >= MONSOON_THRESHOLD_PCT).astype(int)
    features["poor_monsoon_flag"] = (features["rain_anom"] <= -MONSOON_THRESHOLD_PCT).astype(int)
    features["normal_monsoon_flag"] = (
        (features["rain_anom"] > -MONSOON_THRESHOLD_PCT) &
        (features["rain_anom"] < MONSOON_THRESHOLD_PCT)
    ).astype(int)

# Portfolio allocation interpretation target.
features["target_direction"] = (features["next_q_excess_ret"] > 0).astype("Int64")
features["allocation_signal_label"] = np.where(
    features["next_q_excess_ret"] > 0,
    "Mid-cap outperformed large-cap next quarter",
    "Large-cap outperformed or matched mid-cap next quarter",
)
features.loc[features["next_q_excess_ret"].isna(), "allocation_signal_label"] = np.nan

# Date labels.
features["quarter_end"] = features.index
features["year"] = features.index.to_period("Q").year
features["quarter"] = features.index.to_period("Q").quarter
features["quarter_label"] = [f"{y}-Q{q}" for y, q in zip(features["year"], features["quarter"])]

front_cols = ["quarter_end", "year", "quarter", "quarter_label"]
remaining_cols = [c for c in features.columns if c not in front_cols]
features = features[front_cols + remaining_cols]

# Clean model-ready dataset: remove rows where target or one-quarter momentum benchmark is unavailable.
clean = features.dropna(subset=["next_q_excess_ret", "lag_excess_ret_1q"]).copy()

print("All feature rows:", len(features))
print("Clean model-ready rows:", len(clean))
print("Clean date range:", clean["quarter_end"].min(), "to", clean["quarter_end"].max())
clean.head()


All feature rows: 42
Clean model-ready rows: 40
Clean date range: 2014-06-30 00:00:00 to 2024-03-31 00:00:00


,quarter_end,year,quarter,quarter_label,midcap_ret,largecap_ret,excess_ret,cpi_yoy,gdp_yoy,repo,repo_chg_bps,rain_anom,next_q_excess_ret,lag_excess_ret_1q,lag_excess_ret_2q,lag_cpi_yoy_1q,lag_gdp_yoy_1q,lag_repo_1q,lag_repo_chg_bps_1q,lag_rain_anom_1q,good_monsoon_flag,poor_monsoon_flag,normal_monsoon_flag,target_direction,allocation_signal_label
quarter_end,,,,,,,,,,,,,,,,,,,,,,,,,
2014-06-30,2014-06-30,2014,2,2014-Q2,28.847192,13.531070,15.316122,7.859486,7.112080,8.00,0.0,69.879102,-1.747418,0.356270,NaN,8.244298,5.922736,8.0,25.0,69.879102,1,0,0,0,Large-cap outperformed or matched mid-cap next...
2014-09-30,2014-09-30,2014,3,2014-Q3,2.896304,4.643723,-1.747418,6.681568,7.592544,8.00,0.0,-7.449626,6.216425,15.316122,0.356270,7.859486,7.112080,8.0,0.0,69.879102,0,1,0,1,Mid-cap outperformed large-cap next quarter
2014-12-31,2014-12-31,2014,4,2014-Q4,10.207737,3.991312,6.216425,4.054538,8.033806,8.00,0.0,-7.449626,0.802069,-1.747418,15.316122,6.681568,7.592544,8.0,0.0,-7.449626,0,1,0,1,Mid-cap outperformed large-cap next quarter
2015-03-31,2015-03-31,2015,1,2015-Q1,3.316950,2.514880,0.802069,5.272440,7.197433,7.50,-50.0,-7.449626,1.507313,6.216425,-1.747418,4.054538,8.033806,8.0,0.0,-7.449626,0,1,0,1,Mid-cap outperformed large-cap next quarter
2015-06-30,2015-06-30,2015,2,2015-Q2,0.064609,-1.442704,1.507313,5.090809,9.088683,7.25,-25.0,-7.449626,4.820723,0.802069,6.216425,5.272440,7.197433,7.5,-50.0,-7.449626,0,1,0,1,Mid-cap outperformed large-cap next quarter


In [11]:
# =========================
# 9) Data quality checks
# =========================

required_cols = [
    "quarter_end",
    "quarter_label",
    "excess_ret",
    "next_q_excess_ret",
    "lag_excess_ret_1q",
]

missing_required = [c for c in required_cols if c not in clean.columns]
if missing_required:
    raise AssertionError(f"Missing required columns in clean dataset: {missing_required}")

# Validate unique quarters and chronological order.
if clean["quarter_end"].duplicated().any():
    dupes = clean.loc[clean["quarter_end"].duplicated(), "quarter_end"].tolist()
    raise AssertionError(f"Duplicate quarter_end values found: {dupes}")

if not clean["quarter_end"].is_monotonic_increasing:
    raise AssertionError("quarter_end is not sorted in increasing order.")

# Missingness summary.
missing_summary = (
    clean.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda d: d["missing_count"] / len(clean) * 100.0)
    .sort_values(["missing_count", "missing_pct"], ascending=False)
)

save_standard(missing_summary.reset_index().rename(columns={"index": "variable"}), "featurebuilder_missingness_summary.csv")

print("Data quality checks passed.")
print("Missingness summary:")
display(missing_summary.head(25))


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\featurebuilder_missingness_summary.csv
Data quality checks passed.
Missingness summary:


,missing_count,missing_pct
lag_excess_ret_2q,1,2.5
quarter_end,0,0.0
year,0,0.0
quarter,0,0.0
quarter_label,0,0.0
midcap_ret,0,0.0
largecap_ret,0,0.0
excess_ret,0,0.0
cpi_yoy,0,0.0
gdp_yoy,0,0.0


In [12]:
# =========================
# 10) Export final datasets
# =========================

# Full feature table keeps edge rows with lag/target gaps for auditability.
full_csv = PROC / "quarterly_features.csv"
clean_csv = PROC / "quarterly_features_clean.csv"
parquet_path = PROC / "quarterly_features.parquet"

features.to_csv(full_csv, index=False, float_format="%.6f")
clean.to_csv(clean_csv, index=False, float_format="%.6f")

try:
    clean.to_parquet(parquet_path, index=False)
    parquet_msg = str(parquet_path.resolve())
except Exception as exc:
    parquet_msg = f"Parquet not written because the parquet engine is unavailable or failed: {exc}"

print("FeatureBuilder3.0 outputs written:")
print(" -", full_csv.resolve())
print(" -", clean_csv.resolve())
print(" -", parquet_msg)
print("\nMAIN MODELBUILDER INPUT:")
print(clean_csv.resolve())


FeatureBuilder3.0 outputs written:
 - C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_features.csv
 - C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_features_clean.csv
 - C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_features.parquet

MAIN MODELBUILDER INPUT:
C:\Users\Local User\Documents\GitHub\marketpredict\processed\quarterly_features_clean.csv


In [13]:
# =========================
# 11) Export data dictionary
# =========================

def add_def(variable, definition, role):
    return {"variable": variable, "definition": definition, "role": role}

rows = [
    add_def("quarter_end", "Calendar quarter-end date used as the time index.", "time index"),
    add_def("year", "Calendar year derived from quarter_end.", "time label"),
    add_def("quarter", "Calendar quarter number derived from quarter_end.", "time label"),
    add_def("quarter_label", "Readable quarter label in YYYY-Q# format.", "time label"),
    add_def("midcap_ret", "Current-quarter mid-cap index return in percentage terms.", "equity return"),
    add_def("largecap_ret", "Current-quarter large-cap index return in percentage terms.", "equity return"),
    add_def("excess_ret", "Current-quarter mid-cap return minus large-cap return.", "relative return"),
    add_def("next_q_excess_ret", "Next-quarter mid-cap return minus large-cap return; primary dependent variable.", "target"),
    add_def("lag_excess_ret_1q", "One-quarter lagged relative return; core momentum benchmark predictor.", "benchmark predictor"),
    add_def("lag_excess_ret_2q", "Two-quarter lagged relative return; optional momentum predictor.", "predictor"),
    add_def("cpi_yoy", "Consumer price inflation measured as year-over-year percentage change.", "macroeconomic predictor"),
    add_def("gdp_yoy", "GDP growth measured as year-over-year percentage change.", "macroeconomic predictor"),
    add_def("repo", "RBI repo rate level at quarter end.", "policy-rate predictor"),
    add_def("repo_chg_bps", "Quarterly change in repo rate measured in basis points.", "policy-rate predictor"),
    add_def("rain_anom", "Southwest monsoon rainfall anomaly percentage versus normal rainfall.", "climate predictor"),
    add_def("good_monsoon_flag", "Equals 1 when rain_anom is at least the configured positive threshold.", "RQ2 flag"),
    add_def("poor_monsoon_flag", "Equals 1 when rain_anom is at most the configured negative threshold.", "RQ2 flag"),
    add_def("normal_monsoon_flag", "Equals 1 when rain_anom lies between the good and poor monsoon thresholds.", "RQ2 flag"),
    add_def("target_direction", "Equals 1 when next_q_excess_ret is positive; otherwise 0 when target is available.", "allocation classification target"),
    add_def("allocation_signal_label", "Readable interpretation of whether mid-cap or large-cap exposure performed better next quarter.", "interpretation label"),
]

# Add generated lag variables not explicitly listed above.
existing = {r["variable"] for r in rows}
for col in clean.columns:
    if col.startswith("lag_") and col not in existing:
        base_name = col.replace("lag_", "").replace("_1q", "")
        rows.append(add_def(col, f"One-quarter lagged value of {base_name}.", "lagged predictor"))

data_dictionary = pd.DataFrame(rows)
data_dictionary.to_csv(PROC / "data_dictionary.csv", index=False)

print("Saved", (PROC / "data_dictionary.csv").resolve())
display(data_dictionary)


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\data_dictionary.csv


,variable,definition,role
0,quarter_end,Calendar quarter-end date used as the time index.,time index
1,year,Calendar year derived from quarter_end.,time label
2,quarter,Calendar quarter number derived from quarter_end.,time label
3,quarter_label,Readable quarter label in YYYY-Q# format.,time label
4,midcap_ret,Current-quarter mid-cap index return in percen...,equity return
5,largecap_ret,Current-quarter large-cap index return in perc...,equity return
6,excess_ret,Current-quarter mid-cap return minus large-cap...,relative return
7,next_q_excess_ret,Next-quarter mid-cap return minus large-cap re...,target
8,lag_excess_ret_1q,One-quarter lagged relative return; core momen...,benchmark predictor
9,lag_excess_ret_2q,Two-quarter lagged relative return; optional m...,predictor


In [14]:
# =========================
# 12) Export manifest for audit trail
# =========================

manifest = {
    "notebook": "FeatureBuilder3.0.ipynb",
    "purpose": "Build quarterly research dataset for MBA Finance study on monsoon, macroeconomic indicators, and mid-cap versus large-cap relative equity performance in India.",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "raw_folder": str(RAW.resolve()),
    "processed_folder": str(PROC.resolve()),
    "raw_files": RAW_FILES,
    "monsoon_threshold_pct": MONSOON_THRESHOLD_PCT,
    "main_modelbuilder_input": "processed/quarterly_features_clean.csv",
    "target_variable": "next_q_excess_ret",
    "benchmark_variable": "lag_excess_ret_1q",
    "research_question_mapping": {
        "RQ1": "Compare momentum benchmark against enriched macroeconomic and monsoon models using next_q_excess_ret.",
        "RQ2": "Use good_monsoon_flag, poor_monsoon_flag, and next_q_excess_ret to assess monsoon-condition differences.",
        "RQ3": "Use rain_anom and gdp_yoy, including lagged versions, to examine rainfall-growth relationship.",
        "RQ4": "Use model coefficients and permutation importance in ModelBuilder to assess practical allocation relevance."
    },
    "outputs": sorted([p.name for p in PROC.glob("*.csv")]) + (["quarterly_features.parquet"] if (PROC / "quarterly_features.parquet").exists() else []),
    "row_counts": {
        "quarterly_features": int(len(features)),
        "quarterly_features_clean": int(len(clean)),
    },
    "date_range_clean": {
        "start": str(pd.to_datetime(clean["quarter_end"]).min().date()) if len(clean) else None,
        "end": str(pd.to_datetime(clean["quarter_end"]).max().date()) if len(clean) else None,
    },
    "columns_clean": list(clean.columns),
}

manifest_path = PROC / "featurebuilder_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("Saved", manifest_path.resolve())
print(json.dumps(manifest, indent=4)[:2000])


Saved C:\Users\Local User\Documents\GitHub\marketpredict\processed\featurebuilder_manifest.json
{
    "notebook": "FeatureBuilder3.0.ipynb",
    "purpose": "Build quarterly research dataset for MBA Finance study on monsoon, macroeconomic indicators, and mid-cap versus large-cap relative equity performance in India.",
    "created_at": "2026-05-10T23:19:07",
    "raw_folder": "C:\\Users\\Local User\\Documents\\GitHub\\marketpredict\\raw",
    "processed_folder": "C:\\Users\\Local User\\Documents\\GitHub\\marketpredict\\processed",
    "raw_files": {
        "cpi": "CPI_Monthly_Jan_2013_to_Jun_2025.csv",
        "gdp": "GDP_Quarterly_2010_2025.csv",
        "repo": "Repo_Rate_Monthly_2010_2025.csv",
        "midcap": "NIFTYMidcap100.csv",
        "largecap": "Nifty50.csv",
        "rainfall": "AnnualRainfall_with_Good_and_Anomaly_2012_2025.csv"
    },
    "monsoon_threshold_pct": 5.0,
    "main_modelbuilder_input": "processed/quarterly_features_clean.csv",
    "target_variable": "next_q_

## Next step

Use this file as the primary input to ModelBuilder3.0:

`processed/quarterly_features_clean.csv`

FeatureBuilder3.0 intentionally does **not** create RQ-specific result files such as `rq1_metrics.csv`, `rq2_good_vs_poor_rain.csv`, or `rq4_permutation_importance.csv`. Those files belong in ModelBuilder3.0 because they are modeling and analysis outputs, not feature-building outputs.
